In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_data
from shufflenet_v2 import ShuffleNetV2

from kd_utils import student_training_loop, evaluate
from helper_utils import count_params


In [2]:
train_loader, val_loader = load_train_val_data(batch_size=32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## ShuffleNetV2 as a student model with knowledge distillation

In [3]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet_201_teacher.pth"))

<All keys matched successfully>

In [4]:
shufflenet_v2_student = ShuffleNetV2(n_class=10, model_size="0.90x")

model size is  0.90x


In [5]:
count_params(shufflenet_v2_student)

Total Parameters:         1,191,708
Trainable Parameters:     1,176,232
Non-trainable Parameters: 15,476


In [6]:
epochs = 15

In [7]:
shufflenet_v2_student_loss = nn.CrossEntropyLoss()
shufflenet_v2_student_optimizer = optim.Adam(shufflenet_v2_student.parameters(), lr=0.01)

lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(shufflenet_v2_student_optimizer, T_max=epochs, eta_min=0.001)

In [8]:
trained_shufflenet_v2_student, shufflenet_v2_student_history = student_training_loop(
   teacher=densenet_201_teacher,
   student=shufflenet_v2_student,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=shufflenet_v2_student_optimizer,
    temperature=4,
    alpha=0.90,
    num_epochs=epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path= "shufflenet_v2_student_extended.pth",
)

Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 33.7190%, Train Hard Loss: 1.9779, Train Soft Loss: 0.9918, Train Distill Loss: 3.3670 | Val Hard Loss: 2.4690, Val Soft Loss: 0.8337, Val Distill Loss: 7.9037, Val Acc: 36.8106%
Epoch 2/15 - Train Acc: 43.8317%, Train Hard Loss: 1.6791, Train Soft Loss: 0.8543, Train Distill Loss: 2.8780 | Val Hard Loss: 2.1581, Val Soft Loss: 0.7281, Val Distill Loss: 6.9036, Val Acc: 43.1055%
Epoch 3/15 - Train Acc: 49.8873%, Train Hard Loss: 1.5237, Train Soft Loss: 0.7662, Train Distill Loss: 2.5973 | Val Hard Loss: 2.3941, Val Soft Loss: 0.7128, Val Distill Loss: 6.8994, Val Acc: 44.4844%
Epoch 4/15 - Train Acc: 56.7243%, Train Hard Loss: 1.3349, Train Soft Loss: 0.6671, Train Distill Loss: 2.2688 | Val Hard Loss: 1.3808, Val Soft Loss: 0.5397, Val Distill Loss: 5.0078, Val Acc: 57.6739%
Epoch 5/15 - Train Acc: 62.7799%, Train Hard Loss: 1.1351, Train Soft Loss: 0.5667, Train Distill Loss: 1.9283 | Val Hard Loss: 1.5619, Val Soft Loss: 0.5436, Val Distill Loss: 5.1300, Val